# Passo 6 – Grupo de controle: textos verdadeiros do tema eleitoral

**Problema:** o conjunto eleitoral (FactPolCheckBr) só tem boatos falsos. Sem textos verdadeiros do mesmo tipo, não dá para medir **falsos positivos** no domínio eleitoral. O recall de 86% do Passo 4 fica sem contraponto.

**Solução:** montar **dois grupos de controle** com o que já existe, sem inventar dados:

| Controle | O que é | Pontos fortes | Pontos fracos |
|---|---|---|---|
| **C1** | Notícias **verdadeiras** do Fake.Br (parte de teste) e da FakeRecogna que falam de **eleição**, cortadas nas primeiras 40 palavras | Muitas amostras; verdadeiras de fato | São notícias, não mensagens. O estilo é diferente do boato |
| **C2** | Falas e textos **autênticos** que a revisão descartou por não serem o boato (fala real, conteúdo original reaproveitado pelo boato) | Mesmo tema, mesma época e formato curto | Poucas amostras. São "autênticos", não necessariamente "verdadeiros" em tudo. Vêm das matérias de checagem |

**Como ler:** compare o **recall nos boatos** com a **taxa de falsos positivos nos controles**. Se o modelo acerta 86% dos boatos mas também marca 40% dos controles como falsos, ele não discrimina bem.

**Atenção:** nenhum dos controles é perfeito. Declare isso na metodologia.

**Pré-requisito:** notebooks 00 a 04 e `02b` executados.

In [ ]:
import os, re, unicodedata, json, ast, glob, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')
PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
REV = os.path.join(RES, 'revisao')
SEED = 42
K = 40  # corte em palavras, igual ao Passo 5

fake = pd.read_csv(os.path.join(PROC, 'fakebr_prep.csv'))
reco = pd.read_csv(os.path.join(PROC, 'recogna_prep.csv'))
fpc = pd.read_csv(os.path.join(PROC, 'fpc_prep.csv'))
for d in (fake, reco):
    d['rotulo'] = d['rotulo'].astype(int)
tr = fake[fake.particao == 'treino'].reset_index(drop=True)
te = fake[fake.particao == 'teste'].reset_index(drop=True)

with open(os.path.join(RES, 'melhores_parametros.json')) as f:
    PARAMS = {k.split('|')[0]: {p: ast.literal_eval(v) for p, v in d.items()}
              for k, d in json.load(f).items() if k.endswith('sem_stem')}

In [ ]:
import nltk
# O Colab apaga estes recursos quando a sessão acaba, então baixamos de novo (rápido)
for recurso in ['stopwords', 'rslp']:
    nltk.download(recurso, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'sem', 'nada', 'ninguém'}
STOP = set(stopwords.words('portuguese')) - NEGACOES
STEM = RSLPStemmer()

RE_URL = re.compile(r'(https?://\S+|www\.\S+)', re.I)
RE_MENCAO = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_MARCA = re.compile(r'\(sic\.?\)|\[sic\.?\]|\(\.\.\.\)|\[\.\.\.\]', re.I)
RE_NUM = re.compile(r'\d+([.,]\d+)*')
RE_NAOLETRA = re.compile(r'[^a-zA-ZÀ-ÿ ]')

def normalizar(texto):
    t = str(texto)
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_MARCA.sub(' ', t)
    t = RE_MENCAO.sub(' ', t)
    t = RE_HASHTAG.sub(' ', t)
    t = t.lower()
    t = RE_NUM.sub(' num ', t)
    t = RE_NAOLETRA.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenizar(texto):
    return [p for p in normalizar(texto).split() if len(p) > 1 or p == 'e']

def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOP]

def preparar(texto):
    toks = sem_stopwords(tokenizar(texto))
    return ' '.join(toks), ' '.join(STEM.stem(t) for t in toks)

def preparar_proc(texto):
    return ' '.join(sem_stopwords(tokenizar(texto)))

def cortar(serie, k=K):
    return serie.fillna('').str.split().str[:k].str.join(' ')

def novo_modelo(nome):
    tf = TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True)
    clf = MultinomialNB() if nome == 'Naive Bayes' else LinearSVC(random_state=SEED, max_iter=5000)
    p = Pipeline([('tfidf', tf), ('clf', clf)])
    p.set_params(**PARAMS[nome])
    return p

## 1. Montando os controles

**C1:** textos verdadeiros (rótulo 0) cujo texto contém uma palavra do tema eleitoral. A lista de palavras abaixo é **simples e declarada**; pode ser criticada, então fica registrada na metodologia.

In [ ]:
TEMA = re.compile(r'elei[çc][õo]|eleitor|\burnas?\b|\bvoto|votar|votação|candidat|\bTSE\b|presidenci|campanha eleitoral|segundo turno|primeiro turno', re.I)

c1_fake = te[(te.rotulo == 0) & te['texto'].str.contains(TEMA, na=False)].copy()
c1_reco = reco[(reco.rotulo == 0) & reco['texto'].str.contains(TEMA, na=False)].copy()
print('C1 – verdadeiras do Fake.Br (teste) com tema eleitoral:', len(c1_fake))
print('C1 – verdadeiras da FakeRecogna com tema eleitoral:', len(c1_reco))

# C2: falas/textos autênticos descartados na revisão
decisoes = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(REV, 'revisao_claude_lote_*.csv')))])
cand = pd.read_csv(os.path.join(REV, 'revisao_completa.csv'))[['id', 'agencia', 'trecho']]
AUTENTICO = ['fala ou texto verdadeiro', 'conteúdo original reaproveitado', 'fala real', 'tuíte original', 'fala verdadeira']
mask = (decisoes.decisao == 'descartar') & decisoes.motivo.fillna('').apply(lambda m: any(a in m for a in AUTENTICO))
c2 = decisoes[mask][['id', 'motivo']].merge(cand, on='id')
c2 = c2.rename(columns={'trecho': 'texto'})
print('C2 – falas/textos autênticos da revisão:', len(c2), '| mediana de palavras:', c2.texto.str.split().str.len().median())
c2.head(5)

In [ ]:
# Pré-processa: corte em 40 palavras (formato curto) e versão completa
for d in (c1_fake, c1_reco, c2, fpc):
    d['proc_curto'] = [preparar_proc(t) for t in cortar(d['texto'])]
    d['proc_cheio'] = [preparar_proc(t) for t in d['texto'].fillna('')]
tr['proc_curto'] = [preparar_proc(t) for t in cortar(tr['texto'])]
GRUPOS = {'C1 verdadeiras Fake.Br (teste, tema eleitoral)': c1_fake,
          'C1 verdadeiras FakeRecogna (tema eleitoral)': c1_reco,
          'C2 falas autênticas (revisão)': c2}
for nome, g in GRUPOS.items():
    print(f'{nome}: n={len(g)}, mediana de palavras {g.texto.str.split().str.len().median():.0f}')

## 2. Resultado: boatos × controles

Para cada modelo e cada tipo de treino (**texto completo** e **cortado em 40 palavras**):

- **recall nos boatos** = parte dos boatos marcada como falsa;
- **FP nos controles** = parte dos controles verdadeiros marcada como falsa;
- **diferença** = recall − FP. Quanto maior, melhor separa. Zero significa que o modelo não distingue.

Intervalos: Wilson, 95%.

In [ ]:
def ic(k, n):
    lo, hi = proportion_confint(k, n, method='wilson')
    return round(lo, 3), round(hi, 3)

linhas = []
for nome in ['SVM', 'Naive Bayes']:
    for treino, col_tr, col_av in [('texto completo', 'texto_proc', 'proc_cheio'), (f'cortado em {K} palavras', 'proc_curto', 'proc_curto')]:
        m = novo_modelo(nome).fit(tr[col_tr], tr['rotulo'])
        p_boato = m.predict(fpc[col_av]).mean()
        lo_b, hi_b = ic(int(m.predict(fpc[col_av]).sum()), len(fpc))
        for gnome, g in GRUPOS.items():
            pred = m.predict(g[col_av])
            fp = pred.mean()
            lo, hi = ic(int(pred.sum()), len(g))
            linhas.append({'modelo': nome, 'treino_e_avaliacao': treino, 'controle': gnome, 'n_controle': len(g),
                           'recall_boatos': round(p_boato, 3), 'ic_boatos': f'{lo_b}–{hi_b}',
                           'FP_controle': round(fp, 3), 'ic_controle': f'{lo}–{hi}',
                           'diferenca': round(p_boato - fp, 3)})
quadro_controle = pd.DataFrame(linhas)
quadro_controle

## 3. Erros de exemplo

Controles que o melhor modelo marcou como **falsos** (falsos positivos). Leia alguns: eles mostram o que o modelo confunde com desinformação.

In [ ]:
m = novo_modelo('SVM').fit(tr['proc_curto'], tr['rotulo'])
for gnome, g in GRUPOS.items():
    sub = g[m.predict(g['proc_curto']) == 1]
    print(f'\n== {gnome}: {len(sub)} de {len(g)} marcados como falsos')
    for t in sub['texto'].head(5):
        print(' -', ' '.join(str(t).split()[:35]))

## Salvando

In [ ]:
quadro_controle.to_csv(os.path.join(RES, 'quadro10_controle_verdadeiros.csv'), index=False)
c2[['id', 'agencia', 'motivo']].to_csv(os.path.join(RES, 'controle_c2_ids.csv'), index=False)
print('Resultados salvos em', RES)